# 02 · Federated FedAvg + Gate G4  (Phase **P2** — the make-or-break gate)

This is the phase that would have caught the old project's collapse on day one. Before we study
*any* heterogeneity, we prove the federated loop is **correct**:

> **Gate G4 — FedAvg on IID data must reach ≈ the centralized accuracy (95.7%).**
> If it doesn't, the FL loop is broken and nothing downstream can be trusted.

**Design (why two layers).** The Flower simulation API has changed a lot across versions, and we
can't have the make-or-break gate depend on an untested framework version. So:
1. **Transparent FedAvg core** (`fedsat.fl.run_fedavg`) — a correct reference FedAvg
   (SGD clients, few local epochs) that evaluates the **global** model every round on the held-out
   test set (fixes old bugs B4/B5). **This clears G4.**
2. **Real Flower parity** (last section, optional) — reruns the *same* FedAvg through the Flower
   framework and checks the numbers agree, so the thesis can cite Flower. It's version-pinned and
   defensive: if your Colab's Flower API differs, it degrades gracefully and the core result stands.

> Run `00` and `01` first. Then `Runtime > Run all`.


## 1 · Environment + Drive

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/sgogoigh/Satellite-Image-Classification-Fed-Learning.git"
REPO_DIR = "/content/Satellite-Image-Classification-Fed-Learning"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import torch, fedsat
from fedsat.utils import get_device
print("fedsat", fedsat.__version__, "| torch", torch.__version__, "| device:", get_device())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.")


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/fedsat"
except Exception as e:
    print("Drive not mounted (", e, ") -> falling back to local ./artifacts")
    DRIVE = os.path.join(REPO_DIR, "artifacts")
os.makedirs(DRIVE, exist_ok=True)
print("Artifacts dir:", DRIVE)


## 2 · Configuration — **IID** setting (α = 100)

G4 is an *IID* check, so we use a high Dirichlet α (≈ uniform class distribution per client). Same
backbone/optimizer as the centralized baseline so the only thing that changes is *federation*.
Budget note: with `local_epochs=1`, each round ≈ one full pass over the (distributed) data, so
`num_rounds` is roughly comparable to centralized epochs.


In [ ]:
from fedsat.config import ExperimentConfig
from fedsat.utils import set_seed, get_device

cfg = ExperimentConfig(
    experiment_name="p2_fedavg_iid",
    dataset="eurosat_rgb", hf_repo="blanchon/EuroSAT_RGB",
    num_clients=10, alpha=100.0, seed=42,       # alpha=100 => IID (the G4 setting)
    input_size=64,
    backbone="resnet18", pretrained=True, norm="bn",
    optimizer="sgd", lr=0.01, momentum=0.9, weight_decay=5e-4, batch_size=64,
    num_rounds=25, local_epochs=1, fraction_fit=1.0,
    num_workers=2,
    data_cache_dir=f"{DRIVE}/hf_cache",
    partition_dir=f"{DRIVE}/partitions",
    results_dir=f"{DRIVE}/results",
    device=get_device(),
)
set_seed(cfg.seed)
print(cfg)


## 3 · Load data + build the IID partition

In [ ]:
from fedsat.data import (load_eurosat, integrity_gate, build_partition,
                         save_partition, load_partition, partition_matrix)
import numpy as np

hf_ds, class_names, labels = load_eurosat(cfg.hf_repo, cache_dir=cfg.data_cache_dir)
stats = integrity_gate(class_names, labels, expected_classes=cfg.num_classes,
                       expected_total=cfg.expected_total)

try:
    partition = load_partition(cfg)
    print("loaded existing IID partition:", cfg.partition_path())
except FileNotFoundError:
    partition = build_partition(cfg, labels, class_names, data_hash=stats["data_hash"])
    save_partition(cfg, partition)
    print("built + saved IID partition:", cfg.partition_path())

mat = partition_matrix(partition, cfg.num_classes, labels)
print("per-client train class counts (rows=class, cols=client) -- should be ~uniform at alpha=100:")
print(mat)
print("global_test size:", len(partition["global_test"]))


## 4 · Run FedAvg (transparent core) — global model evaluated every round (G5)

Each round: broadcast the global weights → every client does `local_epochs` of SGD on its own data
→ sample-weighted average → **evaluate the aggregated global model on the global test set**. Expect
the per-round accuracy to climb toward ~0.95. On a T4 this is roughly one epoch of compute per round
(~25 rounds ≈ 30–45 min); lower `num_rounds` to go faster.


In [ ]:
from fedsat.fl import run_fedavg

print(f"FedAvg | K={cfg.num_clients} clients | {cfg.num_rounds} rounds x {cfg.local_epochs} "
      f"local epoch | fraction_fit={cfg.fraction_fit} | IID (alpha={cfg.alpha})\n")
global_model, history, summary = run_fedavg(
    cfg, hf_ds, partition, class_names,
    num_rounds=cfg.num_rounds, local_epochs=cfg.local_epochs, fraction_fit=cfg.fraction_fit)

print("\nbest round:", summary["best_round"],
      "| best global-test acc:", round(summary["best_accuracy"], 4),
      "| total comm:", round(summary["total_comm_mb"], 1), "MB")


## 5 · **Gate G4** — FedAvg (IID) vs centralized

In [ ]:
import os, json

# centralized baseline from P1 (fall back to the recorded value if the file isn't on this Drive)
cent_acc = 0.9573
p1_metrics = f"{DRIVE}/results/p1_centralized/metrics.json"
if os.path.exists(p1_metrics):
    cent_acc = json.load(open(p1_metrics))["metrics"]["accuracy"]

fed_acc = summary["best_accuracy"]
gap = cent_acc - fed_acc
TOL = 0.03
print(f"centralized (P1)          global-test acc : {cent_acc:.4f}")
print(f"FedAvg IID (best round)   global-test acc : {fed_acc:.4f}")
print(f"gap (centralized - FedAvg)                : {gap:+.4f}")
if fed_acc >= cent_acc - TOL:
    print(f"\nPASSED G4 -- FedAvg is within {TOL:.0%} of centralized on IID data. The FL loop is CORRECT.")
    print("Cleared to proceed to P3 (non-IID heterogeneity sweep).")
else:
    print(f"\nG4 NOT met yet (gap {gap:.3f} > {TOL}). Before proceeding, increase num_rounds "
          f"(e.g. 40) and/or local_epochs=2, or inspect the FL loop. Do NOT move to non-IID until G4 is green.")


## 6 · Convergence + communication

In [ ]:
import matplotlib.pyplot as plt

r = [h["round"] for h in history]
acc = [h["accuracy"] for h in history]
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(r, acc, "-o", color="#d35400", label="FedAvg global-test acc")
ax1.axhline(cent_acc, ls="--", color="#444", label=f"centralized ({cent_acc:.3f})")
ax1.set_xlabel("federated round"); ax1.set_ylabel("global-test accuracy"); ax1.set_ylim(0, 1)
ax1.legend(loc="lower right"); ax1.grid(alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(r, [h["comm_mb_cumulative"] for h in history], color="gray", alpha=0.4)
ax2.set_ylabel("cumulative communication (MB)", color="gray")
plt.title("FedAvg convergence (IID) vs centralized upper bound")
plt.tight_layout(); plt.show()


## 7 · Final metrics + save

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, torch, os
from fedsat.utils import save_json, git_commit, utc_now

m = summary["final_metrics"]
print("FedAvg final global-test:  acc", round(m["accuracy"], 4),
      "| macro-F1", round(m["macro_f1"], 4), "| kappa", round(m["cohen_kappa"], 4))

cm = np.array(m["confusion_matrix"])
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("FedAvg (IID) -- confusion (global test)")
plt.tight_layout(); plt.show()

run = cfg.run_dir()
cfg.to_yaml(os.path.join(run, "config.yaml"))
save_json(os.path.join(run, "metrics.json"), {
    "regime": "fedavg_iid",
    "summary": {k: v for k, v in summary.items() if k != "final_metrics"},
    "final_metrics": m, "history": history, "centralized_acc": cent_acc,
    "data_hash": partition["meta"]["data_hash"], "git_commit": git_commit(), "created_at": utc_now(),
})
torch.save({"state_dict": global_model.state_dict(), "config": cfg.__dict__},
           os.path.join(run, "fedavg_iid_resnet18.pt"))
print("saved ->", run)


## 8 · (Optional) Real **Flower** parity check

Reruns FedAvg through the **Flower** framework (Simulation Engine) for a few rounds and compares the
final global-test accuracy to our transparent core. This is what lets the write-up honestly cite
Flower. It is **version-pinned and defensive**: if the installed Flower API differs from what's coded
here, it prints the error + your `flwr.__version__` and leaves the P2 result untouched — send me that
and I'll finalize this cell for your exact version.

Set `RUN_FLOWER_PARITY = False` to skip.


In [ ]:
RUN_FLOWER_PARITY = True
FLOWER_ROUNDS = 8   # short parity run (not the full schedule)

if RUN_FLOWER_PARITY:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flwr[simulation]==1.13.1"], check=True)
        import flwr as fl
        from flwr.client import ClientApp, NumPyClient
        from flwr.server import ServerApp, ServerAppComponents, ServerConfig
        from flwr.server.strategy import FedAvg as FlwrFedAvg
        from flwr.common import Context, ndarrays_to_parameters
        from flwr.simulation import run_simulation
        import numpy as np, torch, torch.nn as nn
        from fedsat.models import build_model
        from fedsat.engine import build_optimizer, train_one_epoch, evaluate
        from fedsat.data import make_loader
        print("Flower", fl.__version__)

        device = cfg.device
        def _build():
            return build_model(cfg.backbone, cfg.num_classes, cfg.pretrained, cfg.in_channels, cfg.norm).to(device)
        def get_params(m):
            return [v.detach().cpu().numpy() for v in m.state_dict().values()]
        def set_params(m, params):
            sd = {k: torch.tensor(v) for k, v in zip(m.state_dict().keys(), params)}
            m.load_state_dict(sd, strict=True)

        gtest_loader = make_loader(hf_ds, partition["global_test"], cfg, train=False)
        flwr_hist = []

        class FlwrClient(NumPyClient):
            def __init__(self, cid):
                self.model = _build()
                idx = partition["clients"][str(cid)]["train"]
                self.loader = make_loader(hf_ds, idx, cfg, train=True)
                self.n = len(idx)
            def fit(self, params, config):
                set_params(self.model, params)
                opt = build_optimizer(self.model, cfg); crit = nn.CrossEntropyLoss()
                for _ in range(cfg.local_epochs):
                    train_one_epoch(self.model, self.loader, opt, crit, device)
                return get_params(self.model), self.n, {}
            def evaluate(self, params, config):
                return 0.0, self.n, {}

        def client_fn(context: Context):
            pid = int(context.node_config["partition-id"])
            return FlwrClient(pid).to_client()
        client_app = ClientApp(client_fn=client_fn)

        def evaluate_fn(server_round, params, config):
            m = _build(); set_params(m, params)
            res = evaluate(m, gtest_loader, device, cfg.num_classes, class_names)
            flwr_hist.append({"round": server_round, "accuracy": res["accuracy"]})
            return 1.0 - res["accuracy"], {"accuracy": res["accuracy"], "macro_f1": res["macro_f1"]}

        def server_fn(context: Context):
            init = _build()
            strat = FlwrFedAvg(
                fraction_fit=1.0, fraction_evaluate=0.0,
                min_fit_clients=cfg.num_clients, min_available_clients=cfg.num_clients,
                initial_parameters=ndarrays_to_parameters(get_params(init)),
                evaluate_fn=evaluate_fn)
            return ServerAppComponents(strategy=strat, config=ServerConfig(num_rounds=FLOWER_ROUNDS))
        server_app = ServerApp(server_fn=server_fn)

        run_simulation(server_app=server_app, client_app=client_app,
                       num_supernodes=cfg.num_clients,
                       backend_config={"client_resources": {"num_cpus": 2, "num_gpus": 1.0}})

        core_at_R = history[FLOWER_ROUNDS - 1]["accuracy"] if len(history) >= FLOWER_ROUNDS else summary["best_accuracy"]
        flwr_final = flwr_hist[-1]["accuracy"] if flwr_hist else float("nan")
        print(f"\nPARITY  transparent-core acc @ round {FLOWER_ROUNDS} = {core_at_R:.4f} "
              f"| Flower acc @ round {FLOWER_ROUNDS} = {flwr_final:.4f}")
        print("Close agreement confirms our FedAvg matches the Flower framework => we can cite Flower.")
    except Exception as e:
        print("Flower parity SKIPPED (framework/version issue). The transparent-core P2 result stands.")
        try:
            import flwr as _f; print("flwr version:", _f.__version__)
        except Exception:
            print("flwr not importable")
        print("Send me the flwr version + this error and I'll finalize the Flower cell:")
        print(repr(e))


## Done — P2 complete

- ✅ If **G4 passed**, the federated loop is correct (FedAvg ≈ centralized on IID) — the check the
  old project never did. You have a legitimate FedAvg curve + saved results.
- The Flower parity section (if it ran) confirms the framework agrees with our implementation.

**Next — Phase P3 (`03_noniid_sweep.ipynb`):** now that the loop is trusted, introduce heterogeneity
— sweep Dirichlet α ∈ {100, 1.0, 0.5, 0.1} and compare centralized / local-only / FedAvg / FedProx.
This is where the *interesting* science (and the motivation for the proposed method) begins.
